# t050 — PDE5 문헌으로 만드는 로컬 RAG 파이프라인

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fourmodern/2026_aidrugdiscovery/blob/main/Day06_LLM_Agent/t050_simple-local-rag.ipynb)

**Day 06 · LLM 활용 & RAG 실습**

PDF 논문 여섯 편을 내려받아 **검색 가능한 지식 베이스**로 바꾸고,
질문이 들어오면 근거 문단을 골라 로컬 LLM에게 건네 **출처가 붙은 답변**을 받는다.
외부 벡터 DB나 LangChain 같은 프레임워크 없이, 표준 라이브러리 조합만으로 전 과정을 직접 조립한다.

## 저작권 및 출처 표기

- 이 노트북은 **본 강의를 위해 처음부터 작성한 독자 구현물**이다.
  코드 구조(설정 dataclass → 수집기 → 텍스트 계층 → 패시지 빌더 → 인덱스 → 프롬프트 → 생성기 파사드),
  변수·함수 명명, 서술 문장은 모두 이 강의용으로 새로 설계했다.
- 다루는 주제인 *Retrieval-Augmented Generation* 은 2020년 Lewis 등이 정식화한 이래
  널리 쓰이는 **표준 기법**이며, 여기 구현된 파이프라인은 특정 저작물의 파생물이 아니다.
  PyMuPDF·spaCy·sentence-transformers·transformers 의 호출 형태가 다른 예제와 닮아 보인다면,
  그것은 각 라이브러리의 **공개 API 사용법이 사실상 하나로 정해져 있기 때문**이다.
- **코퍼스**: Europe PMC 오픈액세스 PDE5 관련 논문 6편.
  각 논문의 라이선스는 아래 1단계에서 Europe PMC REST API 로 **실제 조회하여 표에 출력**한다
  (하드코딩하지 않는다). CC BY 계열이라도 조건은 논문마다 다르므로 재배포 전 표를 확인할 것.
- **모델**: `sentence-transformers/all-mpnet-base-v2` (Apache-2.0),
  `Qwen/Qwen3-1.7B` (Apache-2.0). 대안으로 안내하는 `google/gemma-2b-it` 은
  Gemma Terms of Use 가 적용되는 **gated 모델**이다.

## 이 노트북의 자리 — 6일차 다른 실습과의 관계

이 노트북은 **임베딩 기반 RAG 파이프라인 자체**를 처음부터 만든다.
같은 6일차의 에이전트 실습
[`nb1_scientific_agent.ipynb`](https://github.com/fourmodern/2026_aidrugdiscovery/blob/main/20260905/notebooks/nb1_scientific_agent.ipynb)
에서도 RAG가 등장하지만, 거기서는 RAG를 **간이 TF-IDF로 축소해 LLM-wiki(구조화된 기억)와 대비**하는 용도다.
목적이 다르다 — **여기서는 “검색 품질을 어떻게 만드나”, 거기서는 “검색이냐 기억이냐”**.
두 노트북을 이어서 보면 “검색을 잘 만드는 법”과 “검색을 언제 쓸지 고르는 법”이 짝을 이룬다.

## 학습 목표

이 노트북을 마치면 다음을 스스로 설명하고 재현할 수 있다.

| # | 목표 | 확인 방법 |
|---|------|-----------|
| 1 | PDF에서 **본문만** 뽑아내는 전처리의 필요성 | 머리말·참고문헌을 걸러낸 전후 검색 결과 비교 |
| 2 | 문장 단위 분할 후 **겹침(stride)** 을 준 청킹 설계 | 청크 경계에서 문맥이 잘리지 않음을 확인 |
| 3 | 768차원 임베딩과 **내적 유사도** 검색 | `util.dot_score` → `torch.topk` 상위 k 확인 |
| 4 | 인덱스를 **디스크에 저장하고 다시 불러오기** | 재로딩 후 동일 점수 재현 |
| 5 | 검색 문맥을 **프롬프트에 주입**한 근거 기반 생성 | RAG 답변 vs 무근거 답변 대조 |

### 파이프라인 한눈에 보기

```
Europe PMC (6편)  ──┐
                    │  ① 수집   EuropePmcHarvester
   *.pdf ───────────┤
                    │  ② 추출   PdfTextLayer      (머리말/참고문헌 제거)
   PageText[] ──────┤
                    │  ③ 청킹   PassageBuilder    (10문장 창, stride 8)
   Passage[] ───────┤
                    │  ④ 색인   PassageIndex      (all-mpnet-base-v2, 768d)
   vectors.npz ─────┤
                    │  ⑤ 검색   PassageIndex.query(질의, k)
   Retrieved[] ─────┤
                    │  ⑥ 생성   GroundedPrompt + LocalGenerator
   근거 붙은 답변 ◀──┘
```

---
## 0단계 — 환경 준비

Colab에서는 아래 셀이 필요한 패키지를 설치한다. 로컬에 이미 갖춰져 있다면 자동으로 건너뛴다.

In [ ]:
import importlib, subprocess, sys

REQUIRED = {
    "pymupdf": "pymupdf",                      # PDF 텍스트 레이어
    "spacy": "spacy",                          # 문장 경계 탐지
    "sentence_transformers": "sentence-transformers",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "torch": "torch",
}

def ensure(packages: dict) -> list:
    """import 가능한지 먼저 확인하고, 없는 것만 pip로 설치한다."""
    missing = [pip_name for mod, pip_name in packages.items()
               if importlib.util.find_spec(mod) is None]
    if missing:
        print("설치 대상:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    else:
        print("모든 의존 패키지가 이미 준비되어 있습니다.")
    return missing

ensure(REQUIRED)


import torch

def pick_device() -> str:
    """CUDA가 있으면 GPU, 없으면 CPU. 이 노트북은 CPU에서도 끝까지 실행된다."""
    return "cuda" if torch.cuda.is_available() else "cpu"

DEVICE = pick_device()
print(f"torch      : {torch.__version__}")
print(f"device     : {DEVICE}")
if DEVICE == "cuda":
    print(f"gpu        : {torch.cuda.get_device_name(0)}")
    print(f"vram (GB)  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
else:
    print("gpu        : 없음 → CPU 폴백 (양자화 비활성, float32로 진행)")

### 설정을 한 곳에 모은다

파이프라인 전체가 참조하는 값은 `PipelineConfig` 하나에 담는다.
수강생이 실험할 때 이 셀의 숫자만 바꾸면 뒤 단계가 전부 따라온다.

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path


@dataclass(frozen=True)
class PipelineConfig:
    # 코퍼스
    library_dir: Path = Path("pde5_library")     # 내려받은 PDF 보관 위치
    index_dir: Path = Path("pde5_index")         # 임베딩 인덱스 저장 위치

    # 청킹
    sentences_per_passage: int = 10              # 창 크기: 10문장 = 패시지 1개
    sentence_stride: int = 8                     # 창 이동 폭 → 2문장 겹침
    min_passage_tokens: int = 32                 # 이보다 짧은 조각은 버린다

    # 임베딩 / 검색
    embedder_id: str = "sentence-transformers/all-mpnet-base-v2"   # 768차원
    encode_batch: int = 32
    top_k: int = 5

    # 생성
    generator_id: str = "Qwen/Qwen3-1.7B"
    max_new_tokens: int = 220

    def prepare(self) -> "PipelineConfig":
        self.library_dir.mkdir(parents=True, exist_ok=True)
        self.index_dir.mkdir(parents=True, exist_ok=True)
        return self


CFG = PipelineConfig().prepare()
for key, value in asdict(CFG).items():
    print(f"{key:24s} {value}")

---
## 1단계 — 코퍼스 수집 (Europe PMC)

RAG는 **자기가 읽을 수 있는 문서만큼만 똑똑하다.** 그래서 첫 단계가 코퍼스다.

여기서는 PDE5 억제제(sildenafil, vardenafil …)를 다룬 **오픈액세스 논문 6편**을
Europe PMC 에서 내려받는다. 선택 기준은 세 가지다.

1. **오픈액세스** — 라이선스가 명시되어 강의 자료로 다룰 수 있다.
2. **주제가 겹치되 관점이 다르다** — 혈관신생, 면역, 비뇨기, 부정맥, 뇌혈관.
   관점이 갈려야 검색이 “어느 논문을 골랐는지”가 눈에 보인다.
3. **최신** — 모델의 사전학습 지식만으로는 답하기 어려운 2026년 문헌.

In [ ]:
CORPUS_PMCIDS = [
    "PMC12841899",   # sildenafil & arteriogenesis
    "PMC13024863",   # sildenafil, angiogenesis & immune modulation
    "PMC13076431",   # vardenafil in ED management
    "PMC13149040",   # sildenafil proarrhythmic risk
    "PMC13111112",   # PDE5 inhibitors & cerebral small vessel disease
    "PMC13202541",   # acute sildenafil & arrhythmia susceptibility
]

EPMC_SEARCH = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
EPMC_PDF    = "https://europepmc.org/articles/{pmcid}?pdf=render"

print(f"코퍼스 후보 {len(CORPUS_PMCIDS)}편")

### 서지·라이선스 정보를 먼저 조회한다

PDF를 받기 전에 **서지 메타데이터와 라이선스를 API로 확인**한다.
라이선스를 하드코딩하지 않는 이유는 간단하다 — 값이 바뀔 수 있고,
바뀐 줄 모르고 재배포하는 것이 가장 위험하다.

In [ ]:
import json
import urllib.request
from dataclasses import dataclass


@dataclass
class PaperRef:
    """논문 한 편의 서지 정보 + 로컬 PDF 경로."""
    pmcid: str
    title: str = "(제목 확인 필요)"
    journal: str = "(저널 확인 필요)"
    year: str = "?"
    license: str = "확인 필요"
    pdf_path: "Path | None" = None

    @property
    def label(self) -> str:
        """인용 표기에 쓸 짧은 라벨."""
        return f"{self.pmcid}·{self.year}"


def http_json(url: str, timeout: int = 30) -> dict:
    req = urllib.request.Request(url, headers={"User-Agent": "lecture-rag/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def describe_paper(pmcid: str) -> PaperRef:
    """Europe PMC REST 로 서지·라이선스를 조회한다. 실패해도 예외를 던지지 않는다."""
    url = f"{EPMC_SEARCH}?query={pmcid}&resultType=core&format=json"
    try:
        hits = http_json(url)["resultList"]["result"]
        if not hits:
            return PaperRef(pmcid=pmcid)
        rec = hits[0]
        journal_block = (rec.get("journalInfo") or {}).get("journal") or {}
        return PaperRef(
            pmcid=pmcid,
            title=(rec.get("title") or "(제목 확인 필요)").rstrip("."),
            journal=journal_block.get("title") or rec.get("journalTitle") or "(저널 확인 필요)",
            year=str(rec.get("pubYear") or "?"),
            license=(rec.get("license") or "확인 필요").upper(),
        )
    except Exception as exc:                      # 네트워크·스키마 변경 모두 여기로
        print(f"  ! {pmcid} 메타데이터 조회 실패 ({type(exc).__name__}) → '확인 필요'로 표기")
        return PaperRef(pmcid=pmcid)


CATALOGUE = [describe_paper(pid) for pid in CORPUS_PMCIDS]
print(f"메타데이터 {len(CATALOGUE)}건 조회 완료")

In [ ]:
def render_catalogue(refs: list) -> None:
    """라이선스 표를 사람이 읽을 수 있게 출력한다."""
    width = max(len(r.journal) for r in refs)
    print(f"{'PMCID':<13} {'연도':<5} {'라이선스':<12} {'저널':<{width}}  제목")
    print("-" * (40 + width + 40))
    for r in refs:
        print(f"{r.pmcid:<13} {r.year:<5} {r.license:<12} {r.journal:<{width}}  {r.title[:58]}")


render_catalogue(CATALOGUE)
print()
print("※ 위 라이선스 값은 Europe PMC REST 응답의 `license` 필드를 그대로 옮긴 것이다.")
print("   NC / ND 조건이 붙은 논문은 상업적 재사용·2차 저작에 제한이 있으니 주의할 것.")

### PDF 내려받기 — 실패를 전제로 짠다

여섯 편 중 한 편이 실패해도 실습이 멈추면 곤란하다. 다음을 지킨다.

- 편당 `try/except` — 한 편의 실패가 나머지를 막지 않는다
- **`%PDF` 매직바이트 검사** — 서버가 오류 HTML을 200으로 돌려주는 경우를 잡는다
- **User-Agent 지정** — 기본 파이썬 UA는 차단당하는 일이 있다
- **캐시** — 이미 받아둔 파일은 다시 받지 않는다 (재실행 비용 0)
- 0편이면 **중단**, 3편 미만이면 **경고** — 검색이 무의미해지는 하한선

In [ ]:
PDF_MAGIC = b"%PDF"


class EuropePmcHarvester:
    """PMCID 목록을 받아 로컬 라이브러리에 PDF를 채운다."""

    def __init__(self, config: PipelineConfig):
        self.config = config

    def _target(self, pmcid: str) -> Path:
        return self.config.library_dir / f"{pmcid}.pdf"

    def _download(self, pmcid: str) -> Path:
        target = self._target(pmcid)
        if target.exists() and target.stat().st_size > 0:
            print(f"  · {pmcid} 캐시 사용 ({target.stat().st_size/1e6:.1f} MB)")
            return target

        req = urllib.request.Request(
            EPMC_PDF.format(pmcid=pmcid),
            headers={"User-Agent": "Mozilla/5.0 (compatible; lecture-rag/1.0)"},
        )
        with urllib.request.urlopen(req, timeout=120) as resp:
            payload = resp.read()

        if not payload.startswith(PDF_MAGIC):
            raise ValueError(f"PDF가 아닌 응답 (앞 4바이트 = {payload[:4]!r})")

        target.write_bytes(payload)
        print(f"  + {pmcid} 내려받음 ({len(payload)/1e6:.1f} MB)")
        return target

    def collect(self, refs: list) -> list:
        acquired = []
        for ref in refs:
            try:
                ref.pdf_path = self._download(ref.pmcid)
                acquired.append(ref)
            except Exception as exc:
                print(f"  ! {ref.pmcid} 실패: {type(exc).__name__}: {exc}")
        return acquired

In [ ]:
LIBRARY = EuropePmcHarvester(CFG).collect(CATALOGUE)

print()
if not LIBRARY:
    raise RuntimeError(
        "PDF를 한 편도 확보하지 못했습니다. 네트워크/프록시를 확인한 뒤 다시 실행하세요."
    )
if len(LIBRARY) < 3:
    print(f"[경고] {len(LIBRARY)}편만 확보되었습니다. 검색 다양성이 크게 떨어집니다.")
print(f"확보한 논문 {len(LIBRARY)} / {len(CATALOGUE)}편")

---
## 2단계 — 텍스트 레이어 추출과 정제

PyMuPDF는 PDF 안의 **텍스트 레이어**를 그대로 꺼낸다. 문제는 그 원문이 지저분하다는 것이다.

| 잡음 | 예 | 그대로 두면 |
|------|-----|-------------|
| 러닝 헤더/푸터 | `Frontiers in Neurology 06 frontiersin.org` | 모든 페이지에 반복 → 검색 상위를 잠식 |
| 보이지 않는 문자 | URL 사이의 zero-width joiner | 문자열 검사·정규식이 전부 빗나감 |
| 합자·특수 따옴표 | `ﬁ`, `–`, `’` | 토크나이저가 낯선 토큰으로 처리 |
| 참고문헌 목록 | `[12] Kloner R, et al. …` | 키워드가 빽빽해 **검색 점수를 독식** |

앞의 세 가지는 정규화로, 마지막은 **참고문헌 섹션 절단**으로 처리한다.
이 정제가 RAG 품질에 미치는 영향은 뒤에서 실제 점수로 확인한다.

In [ ]:
import re
import collections

try:
    import pymupdf                      # 1.24+ 권장 이름
except ImportError:                     # 구버전 호환
    import fitz as pymupdf


# 눈에 보이지 않지만 문자열을 망가뜨리는 문자들
INVISIBLE = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u2060\ufeff\u00ad"), None)

# 조판용 글리프 → 평범한 ASCII
GLYPH_FIXES = {
    "\ufb01": "fi", "\ufb02": "fl", "\ufb00": "ff", "\ufb03": "ffi", "\ufb04": "ffl",
    "\u2013": "-", "\u2014": "-", "\u2019": "'", "\u2018": "'",
    "\u201c": '"', "\u201d": '"', "\u00a0": " ",
}


def normalise(raw: str) -> str:
    """보이지 않는 문자 제거 + 조판 글리프 치환."""
    text = raw.translate(INVISIBLE)
    for bad, good in GLYPH_FIXES.items():
        text = text.replace(bad, good)
    return text


def collapse(text: str) -> str:
    """줄바꿈·연속 공백을 단일 공백으로."""
    return re.sub(r"\s+", " ", text).strip()


@dataclass
class PageText:
    """정제를 마친 페이지 한 장."""
    pmcid: str
    page_no: int          # 1부터 시작 (PDF 뷰어와 동일)
    body: str


REFERENCE_HEADING = re.compile(r"\b(References|Bibliography|Literature Cited)\b")

CITATION_MARKS = re.compile(
    r"(et al\.|doi:|doi\.org|CrossRef|PubMed|https?://|\[\s*\d+\s*\])",
    re.IGNORECASE,
)


def _per_100_words(text: str, count: int) -> float:
    return count / max(len(text.split()) / 100, 1)


def looks_like_bibliography(text: str) -> bool:
    """숫자 밀도 · 인용 표지 · 세미콜론 밀도로 참고문헌 블록을 판별한다."""
    if not text:
        return False
    digit_ratio = sum(ch.isdigit() for ch in text) / len(text)
    return (
        digit_ratio > 0.11
        or _per_100_words(text, len(CITATION_MARKS.findall(text))) >= 2.5
        or _per_100_words(text, text.count(";")) >= 5.0
    )

In [ ]:
class PdfTextLayer:
    """PDF 한 편 → 정제된 PageText 목록."""

    #: 몇 %의 페이지에 같은 줄이 나오면 러닝 헤더로 볼 것인가
    RUNNING_LINE_RATIO = 0.4

    @staticmethod
    def _fingerprint(line: str) -> str:
        """쪽번호만 다른 머리말을 같은 것으로 묶기 위해 숫자를 마스킹."""
        return re.sub(r"\d+", "#", line)

    @classmethod
    def _strip_running_lines(cls, pages: list) -> list:
        """여러 페이지에 반복되는 짧은 줄(머리말·꼬리말)을 제거한다."""
        per_page_lines = [[ln.strip() for ln in page.split("\n")] for page in pages]

        tally = collections.Counter()
        for lines in per_page_lines:
            for line in set(lines):
                if 0 < len(line) <= 90:
                    tally[cls._fingerprint(line)] += 1

        threshold = max(2, int(cls.RUNNING_LINE_RATIO * len(pages)))
        boilerplate = {key for key, n in tally.items() if n >= threshold}

        return [
            " ".join(ln for ln in lines if cls._fingerprint(ln) not in boilerplate)
            for lines in per_page_lines
        ]

    @classmethod
    def _reference_cut(cls, pages: list) -> "int | None":
        """참고문헌 섹션이 시작되는 페이지 번호(1-base)를 찾는다."""
        for page_no, text in enumerate(pages, start=1):
            match = REFERENCE_HEADING.search(text)
            if match and looks_like_bibliography(text[match.end():]):
                return page_no
        return None

    @classmethod
    def read(cls, ref: PaperRef, min_chars: int = 120) -> list:
        with pymupdf.open(ref.pdf_path) as document:
            raw_pages = [normalise(page.get_text()) for page in document]

        pages = [collapse(text) for text in cls._strip_running_lines(raw_pages)]
        cut = cls._reference_cut(pages)

        harvested = []
        for page_no, text in enumerate(pages, start=1):
            if cut is not None:
                if page_no > cut:
                    break                                     # 참고문헌 이후 전부 버림
                if page_no == cut:
                    text = text[: REFERENCE_HEADING.search(text).start()]
            if len(text) >= min_chars:
                harvested.append(PageText(ref.pmcid, page_no, text))
        return harvested

In [ ]:
PAGES = []
print(f"{'PMCID':<13} {'본문 페이지':>10}   {'문자 수':>9}")
print("-" * 40)
for ref in LIBRARY:
    pages = PdfTextLayer.read(ref)
    PAGES.extend(pages)
    chars = sum(len(p.body) for p in pages)
    print(f"{ref.pmcid:<13} {len(pages):>10}   {chars:>9,}")

print("-" * 40)
print(f"{'합계':<13} {len(PAGES):>10}   {sum(len(p.body) for p in PAGES):>9,}")


sample = PAGES[len(PAGES) // 2]
print(f"[{sample.pmcid} p.{sample.page_no}] 정제 후 본문 앞부분\n")
print(sample.body[:700], "…")

---
## 3단계 — 패시지 만들기 (문장 분할 → 겹침 창)

**왜 페이지째로 임베딩하지 않는가?**
768차원 벡터 하나가 표현할 수 있는 의미의 양은 정해져 있다.
페이지 전체를 한 벡터로 뭉개면 그 안의 구체적인 주장이 평균값에 묻힌다.
반대로 문장 하나씩 임베딩하면 지시대명사(“이 결과는 …”)의 문맥이 사라진다.
**문장 10개 정도**가 이 균형점이다.

**겹침(stride)을 왜 주는가?**
창을 10문장씩 잘라 붙이면(stride 10) 경계에 걸친 논증이 두 조각으로 갈린다.
stride 를 8로 두면 **연속한 두 패시지가 2문장을 공유**하므로 경계 손실이 줄어든다.
비용은 패시지 수가 약 25% 늘어나는 것뿐이다.

문장 경계 탐지에는 spaCy 의 규칙 기반 `sentencizer` 를 쓴다.
통계 모델(`en_core_web_sm`)이 필요 없어 **추가 다운로드 없이** 동작하고, 이 용도에는 충분히 정확하다.

In [ ]:
from spacy.lang.en import English


class SentenceSplitter:
    """spaCy 규칙 기반 문장 분할기 (모델 다운로드 불필요)."""

    def __init__(self):
        self._nlp = English()
        self._nlp.add_pipe("sentencizer")

    def __call__(self, text: str) -> list:
        return [s.text.strip() for s in self._nlp(text).sents if s.text.strip()]


SPLITTER = SentenceSplitter()

demo = ("Sildenafil selectively inhibits PDE5. This raises intracellular cGMP levels. "
        "Smooth muscle relaxation and vasodilation follow.")
for i, s in enumerate(SPLITTER(demo), 1):
    print(f"{i}. {s}")


@dataclass
class Passage:
    """검색 단위 한 조각."""
    uid: str              # 예: PMC12841899:p08:c02
    pmcid: str
    page_no: int
    body: str
    n_sentences: int

    @property
    def token_estimate(self) -> int:
        """영어 산문에서 대략 4문자 ≈ 1토큰. 길이 필터용 근사치."""
        return len(self.body) // 4

    def citation(self) -> str:
        return f"{self.pmcid} p.{self.page_no}"

In [ ]:
class PassageBuilder:
    """PageText 목록 → 겹침 창 방식의 Passage 목록."""

    def __init__(self, config: PipelineConfig, splitter: SentenceSplitter):
        self.window = config.sentences_per_passage
        self.stride = config.sentence_stride
        self.min_tokens = config.min_passage_tokens
        self.splitter = splitter
        self.rejected = collections.Counter()      # 왜 버렸는지 집계

    def _windows(self, sentences: list):
        for start in range(0, max(len(sentences), 1), self.stride):
            chunk = sentences[start : start + self.window]
            if chunk:
                yield chunk
            if start + self.window >= len(sentences):
                break

    def build(self, pages: list) -> list:
        passages = []
        for page in pages:
            sentences = self.splitter(page.body)
            for order, window in enumerate(self._windows(sentences)):
                body = " ".join(window)
                candidate = Passage(
                    uid=f"{page.pmcid}:p{page.page_no:02d}:c{order:02d}",
                    pmcid=page.pmcid,
                    page_no=page.page_no,
                    body=body,
                    n_sentences=len(window),
                )
                if candidate.token_estimate < self.min_tokens:
                    self.rejected["너무 짧음"] += 1
                elif looks_like_bibliography(body):
                    self.rejected["참고문헌 잔재"] += 1
                else:
                    passages.append(candidate)
        return passages

In [ ]:
BUILDER = PassageBuilder(CFG, SPLITTER)
PASSAGES = BUILDER.build(PAGES)

total_seen = len(PASSAGES) + sum(BUILDER.rejected.values())
print(f"생성된 창       : {total_seen}")
for reason, n in BUILDER.rejected.most_common():
    print(f"  - 제거 ({reason}) : {n}")
print(f"최종 패시지     : {len(PASSAGES)}")


lengths = sorted(p.token_estimate for p in PASSAGES)
per_paper = collections.Counter(p.pmcid for p in PASSAGES)

print("패시지 길이(추정 토큰)")
print(f"  최소 {lengths[0]} / 중앙값 {lengths[len(lengths)//2]} / 최대 {lengths[-1]}")
print(f"  평균 문장 수 {sum(p.n_sentences for p in PASSAGES)/len(PASSAGES):.1f}")
print("\n논문별 패시지 수")
for pmcid, n in per_paper.most_common():
    print(f"  {pmcid}  {'█' * round(n / 3):<20} {n}")

In [ ]:
def first_adjacent_pair(passages: list) -> tuple:
    """같은 페이지 안에서 연속한(c00→c01 …) 패시지 쌍을 찾는다.

    청킹은 페이지 단위로 하므로 리스트에서 그냥 이웃한 두 개를 집으면
    페이지 경계를 넘어 겹침이 0으로 나올 수 있다 — 아래 주의 참고.
    """
    for a, b in zip(passages, passages[1:]):
        if a.pmcid == b.pmcid and a.page_no == b.page_no:
            return a, b
    return passages[0], passages[1]


first, second = first_adjacent_pair(PASSAGES)
print(f"같은 페이지 안에서 연속한 두 패시지: {first.uid} → {second.uid}\n")
print(f"[{first.uid}]  …{first.body[-220:]}")
print()
print(f"[{second.uid}]  {second.body[:220]}…")

shared = set(SPLITTER(first.body)) & set(SPLITTER(second.body))
print(f"\n공유 문장 수: {len(shared)}"
      f"  (stride={CFG.sentence_stride}, window={CFG.sentences_per_passage} → 기대값 2)")

crossings = sum(1 for a, b in zip(PASSAGES, PASSAGES[1:])
                if a.pmcid == b.pmcid and a.page_no != b.page_no)
print(f"\n[주의] 페이지 경계 {crossings}곳에서는 겹침이 없다. "
      f"청킹을 페이지 단위로 하기 때문이다 — 개선 과제로 남겨 둔다.")

---
## 4단계 — 임베딩과 인덱스

`all-mpnet-base-v2` 는 문장/문단을 **768차원 단위벡터**로 사상한다.
정규화된 벡터끼리의 **내적(dot product)** 은 곧 코사인 유사도이므로,
별도 정규화 없이 `util.dot_score` 로 바로 비교할 수 있다.

인덱스는 **디스크에 저장**한다. 임베딩은 이 파이프라인에서 가장 비싼 단계이고,
코퍼스가 바뀌지 않는 한 다시 계산할 이유가 없다.

- `vectors.npz` — float32 행렬 (N × 768). 텍스트와 분리해 두면 로딩이 빠르다
- `passages.jsonl` — 패시지 메타데이터 한 줄 = 한 개. 사람이 열어볼 수 있다
- `manifest.json` — 어떤 모델·설정으로 만들었는지. **이게 없으면 재현이 불가능하다**

In [ ]:
from sentence_transformers import SentenceTransformer, util

ENCODER = SentenceTransformer(CFG.embedder_id, device=DEVICE)

# sentence-transformers 6.x 에서 이름이 바뀐 메서드 — 양쪽 모두 지원
_dim_fn = getattr(ENCODER, "get_embedding_dimension", None) \
          or ENCODER.get_sentence_embedding_dimension
WINDOW_TOKENS = ENCODER.max_seq_length

print(f"임베딩 모델 : {CFG.embedder_id}")
print(f"출력 차원   : {_dim_fn()}")
print(f"최대 입력   : {WINDOW_TOKENS} 토큰")

probe = ENCODER.encode(["sildenafil relaxes vascular smooth muscle"],
                       convert_to_tensor=True)
print(f"\n벡터 shape : {tuple(probe.shape)}")
print(f"L2 norm    : {float(probe.norm()):.4f}  (≈1 이면 정규화된 상태)")
print(f"앞 8개 성분: {[round(float(v), 4) for v in probe[0][:8]]}")

# 모델의 입력 창을 넘는 패시지는 뒷부분이 잘린다 — 알고 쓰는 것과 모르는 것은 다르다
oversize = [p for p in PASSAGES if p.token_estimate > WINDOW_TOKENS]
print(f"\n입력 창({WINDOW_TOKENS} 토큰)을 넘는 패시지: "
      f"{len(oversize)} / {len(PASSAGES)} ({len(oversize)/len(PASSAGES)*100:.0f}%)")
print("  → 초과분은 인코딩 시 절단된다. 창 크기를 줄이거나(예: 6문장) "
      "긴 입력을 받는 모델로 교체하면 완화된다.")

In [ ]:
import numpy as np
import time


class PassageIndex:
    """패시지 + 벡터를 함께 들고 다니는 최소한의 벡터 스토어."""

    def __init__(self, passages: list, matrix: "torch.Tensor", manifest: dict):
        self.passages = passages
        self.matrix = matrix
        self.manifest = manifest

    def __len__(self) -> int:
        return len(self.passages)

    # ---------- 생성 ----------
    @classmethod
    def build(cls, passages: list, encoder: SentenceTransformer,
              config: PipelineConfig) -> "PassageIndex":
        started = time.time()
        matrix = encoder.encode(
            [p.body for p in passages],
            batch_size=config.encode_batch,
            convert_to_tensor=True,
            show_progress_bar=False,
        )
        manifest = {
            "embedder_id": config.embedder_id,
            "dimension": int(matrix.shape[1]),
            "n_passages": len(passages),
            "sentences_per_passage": config.sentences_per_passage,
            "sentence_stride": config.sentence_stride,
            "min_passage_tokens": config.min_passage_tokens,
            "build_seconds": round(time.time() - started, 1),
        }
        return cls(passages, matrix, manifest)

    # ---------- 저장 / 로딩 ----------
    def save(self, directory: Path) -> Path:
        directory.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(directory / "vectors.npz",
                            matrix=self.matrix.cpu().numpy().astype("float32"))
        with (directory / "passages.jsonl").open("w", encoding="utf-8") as fh:
            for p in self.passages:
                fh.write(json.dumps(asdict(p), ensure_ascii=False) + "\n")
        (directory / "manifest.json").write_text(
            json.dumps(self.manifest, ensure_ascii=False, indent=2), encoding="utf-8")
        return directory

    @classmethod
    def load(cls, directory: Path, device: str = "cpu") -> "PassageIndex":
        matrix = torch.from_numpy(np.load(directory / "vectors.npz")["matrix"]).to(device)
        passages = [Passage(**json.loads(line))
                    for line in (directory / "passages.jsonl").read_text(encoding="utf-8").splitlines()]
        manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))
        return cls(passages, matrix, manifest)

    # ---------- 검색 ----------
    def query(self, question: str, encoder: SentenceTransformer, k: int = 5) -> list:
        query_vec = encoder.encode(question, convert_to_tensor=True).to(self.matrix.device)
        scores = util.dot_score(query_vec, self.matrix)[0]
        best = torch.topk(scores, k=min(k, len(self)))
        return [Retrieved(self.passages[int(i)], float(s))
                for s, i in zip(best.values, best.indices)]


@dataclass
class Retrieved:
    passage: Passage
    score: float

In [ ]:
INDEX = PassageIndex.build(PASSAGES, ENCODER, CFG)
print(json.dumps(INDEX.manifest, ensure_ascii=False, indent=2))


saved_to = INDEX.save(CFG.index_dir)
for artefact in sorted(saved_to.iterdir()):
    print(f"{artefact.name:<18} {artefact.stat().st_size/1024:>9,.1f} KB")

In [ ]:
# 저장한 인덱스를 다시 읽어 동일성을 확인한다 — 재현성 점검
RELOADED = PassageIndex.load(CFG.index_dir, device=DEVICE)

print(f"패시지 수 일치 : {len(RELOADED) == len(INDEX)}")
print(f"행렬 shape     : {tuple(RELOADED.matrix.shape)}")
print(f"최대 오차      : {float((RELOADED.matrix - INDEX.matrix).abs().max()):.2e}  (float32 반올림 수준)")

---
## 5단계 — 검색 (Retrieval)

질의도 같은 인코더로 임베딩한 뒤 `util.dot_score` 로 전체 패시지와의 유사도를 구하고
`torch.topk` 로 상위 k개를 뽑는다. 이것이 RAG의 R 전부다.

점수를 읽는 법 — 이 모델·이 코퍼스 기준의 대략적인 감:

| 점수대 | 해석 |
|--------|------|
| 0.70 이상 | 질의를 직접 다루는 문단. 근거로 쓸 만하다 |
| 0.55 ~ 0.70 | 주제는 맞지만 초점이 어긋남. 보조 문맥 |
| 0.55 미만 | 사실상 무관. k를 키워도 얻을 게 없다 |

In [ ]:
def show_hits(hits: list, width: int = 260) -> None:
    for rank, hit in enumerate(hits, 1):
        p = hit.passage
        print(f"[{rank}] score={hit.score:.4f}   {p.citation()}   ({p.n_sentences}문장)")
        print(f"     {p.body[:width]}…")
        print()

In [ ]:
QUERY_MECHANISM = "mechanism of sildenafil as a PDE5 inhibitor"

hits_mechanism = RELOADED.query(QUERY_MECHANISM, ENCODER, k=CFG.top_k)
print(f"질의: {QUERY_MECHANISM}\n")
show_hits(hits_mechanism)

In [ ]:
QUERY_SAFETY = "cardiovascular safety and arrhythmia risk of PDE5 inhibitors"

hits_safety = RELOADED.query(QUERY_SAFETY, ENCODER, k=CFG.top_k)
print(f"질의: {QUERY_SAFETY}\n")
show_hits(hits_safety)


# 두 질의가 실제로 서로 다른 논문을 불러오는지 — 코퍼스 다양성 확인
def sources_of(hits: list) -> str:
    return ", ".join(sorted({h.passage.pmcid for h in hits}))

print(f"기전 질의  → {sources_of(hits_mechanism)}")
print(f"안전성 질의 → {sources_of(hits_safety)}")

### 전처리가 검색에 미친 영향

2단계에서 걸러낸 참고문헌·머리말이 남아 있었다면 어떻게 됐을까?
직접 확인해 보자. 필터를 끄고 인덱스를 다시 만들어 같은 질의를 던진다.

In [ ]:
class NaivePassageBuilder(PassageBuilder):
    """비교용: 길이 필터만 적용하고 참고문헌 판별은 하지 않는다."""

    def build(self, pages: list) -> list:
        passages = []
        for page in pages:
            for order, window in enumerate(self._windows(self.splitter(page.body))):
                body = " ".join(window)
                candidate = Passage(
                    uid=f"{page.pmcid}:p{page.page_no:02d}:c{order:02d}",
                    pmcid=page.pmcid, page_no=page.page_no,
                    body=body, n_sentences=len(window),
                )
                if candidate.token_estimate >= self.min_tokens:
                    passages.append(candidate)
        return passages


# 참고문헌 절단 이전의 '날것' 페이지를 다시 만든다
def raw_pages_of(ref: PaperRef) -> list:
    with pymupdf.open(ref.pdf_path) as document:
        return [PageText(ref.pmcid, i, collapse(normalise(page.get_text())))
                for i, page in enumerate(document, start=1)
                if len(collapse(normalise(page.get_text()))) >= 120]


RAW_PAGES = [p for ref in LIBRARY for p in raw_pages_of(ref)]
NAIVE_PASSAGES = NaivePassageBuilder(CFG, SPLITTER).build(RAW_PAGES)
print(f"정제 파이프라인 : {len(PASSAGES)} 패시지")
print(f"무정제 파이프라인: {len(NAIVE_PASSAGES)} 패시지")


NAIVE_INDEX = PassageIndex.build(NAIVE_PASSAGES, ENCODER, CFG)
naive_hits = NAIVE_INDEX.query(QUERY_SAFETY, ENCODER, k=3)

print(f"[무정제] {QUERY_SAFETY}\n")
show_hits(naive_hits, width=200)
print("=" * 90)
print(f"[정제됨] {QUERY_SAFETY}\n")
show_hits(hits_safety[:3], width=200)

> **관찰하기.** 무정제 인덱스에서는 상위권에 참고문헌 목록이나 판권 페이지가 섞여 들어오기 쉽다.
> 저자명과 저널명이 빽빽한 텍스트는 어떤 질의와도 어중간하게 닮았기 때문이다.
> 점수 자체는 높게 나올 수 있지만 **LLM에게 건네줄 근거로는 쓸모가 없다.**
> RAG를 개선할 때 임베딩 모델을 바꾸기 전에 **전처리부터 손보는 편이 대개 이득**인 이유가 여기 있다.

---
## 6단계 — 근거 기반 생성 (Augmented Generation)

이제 검색된 패시지를 프롬프트에 넣는다. 설계에서 신경 쓸 점 세 가지.

1. **근거에 번호를 붙인다** — `[S1] (PMC… p.8)` 형태로 표시하면
   모델이 `[S1]` 이라고 인용할 수 있고, 사람이 원문으로 되짚어갈 수 있다.
2. **역할을 시스템 메시지로 고정한다** — “주어진 근거만 사용하고, 없으면 없다고 말하라.”
   이 한 줄이 환각을 눈에 띄게 줄인다.
3. **`apply_chat_template` 을 쓴다** — 모델마다 대화 포맷(특수 토큰)이 다르다.
   토크나이저가 아는 형식을 직접 문자열로 짜맞추려 들지 말 것.

Qwen3 는 사고 과정을 길게 출력하는 *thinking* 모드가 기본이라 CPU에서는 답이 늦다.
`enable_thinking=False` 로 꺼 둔다.

In [ ]:
class GroundedPrompt:
    """검색 결과를 근거 블록으로 묶어 chat 메시지를 만든다."""

    SYSTEM = (
        "You are a careful biomedical research assistant. "
        "Answer ONLY from the numbered evidence passages provided. "
        "Cite the passages you rely on using their tags, e.g. [S1] or [S2]. "
        "If the evidence does not cover the question, say so explicitly "
        "instead of guessing. Be concise: at most six sentences."
    )

    @staticmethod
    def evidence_block(hits: list) -> str:
        lines = []
        for rank, hit in enumerate(hits, 1):
            lines.append(f"[S{rank}] ({hit.passage.citation()}) {hit.passage.body}")
        return "\n\n".join(lines)

    @classmethod
    def compose(cls, question: str, hits: list) -> list:
        user = (
            "Evidence passages:\n\n"
            f"{cls.evidence_block(hits)}\n\n"
            "---\n"
            f"Question: {question}\n\n"
            "Answer using only the evidence above, with inline citations."
        )
        return [{"role": "system", "content": cls.SYSTEM},
                {"role": "user", "content": user}]


preview = GroundedPrompt.compose(QUERY_MECHANISM, hits_mechanism[:2])
print(f"메시지 수: {len(preview)}\n")
print("── system ──")
print(preview[0]["content"])
print("\n── user (앞 900자) ──")
print(preview[1]["content"][:900], "…")

### 생성 모델 로드

기본값은 **`Qwen/Qwen3-1.7B`** — Apache-2.0 라이선스라 게이팅이 없고,
CPU에서도 (느리지만) 끝까지 돌아간다.

**`google/gemma-2b-it` 을 쓰고 싶다면**:
Hugging Face 계정으로 [모델 페이지](https://huggingface.co/google/gemma-2b-it)에서 약관에 동의한 뒤
`huggingface-cli login` 또는 `HF_TOKEN` 환경변수로 인증해야 한다.
`CFG.generator_id` 만 바꾸면 나머지 코드는 그대로 동작한다 — 두 모델 모두
`apply_chat_template` 을 지원하기 때문이다.

**dtype·양자화 분기**: GPU가 있으면 bfloat16, 없으면 float32.
4bit 양자화(`bitsandbytes`)는 CUDA 전용이므로 CPU에서는 자동으로 비활성화한다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM


class LocalGenerator:
    """로컬 causal LM 래퍼. GPU/CPU에 따라 dtype·양자화를 알아서 고른다."""

    def __init__(self, model_id: str, device: str):
        self.model_id = model_id
        self.device = device
        self.tokenizer = None
        self.model = None

    def _load_kwargs(self) -> dict:
        if self.device == "cuda":
            kwargs = {"dtype": torch.bfloat16, "device_map": "auto"}
            free_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            if free_gb < 6:
                try:
                    from transformers import BitsAndBytesConfig
                    kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)
                    print("  VRAM 여유가 적어 4bit 양자화를 활성화합니다.")
                except ImportError:
                    print("  bitsandbytes 미설치 → 양자화 없이 진행합니다.")
            return kwargs
        return {"dtype": torch.float32}          # CPU: 양자화 off, float32

    def load(self) -> "LocalGenerator":
        print(f"로드 중: {self.model_id} (device={self.device})")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model = AutoModelForCausalLM.from_pretrained(self.model_id, **self._load_kwargs())
        self.model.eval()
        if self.device == "cpu":
            self.model.to("cpu")
        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"완료 · 파라미터 {n_params/1e9:.2f}B")
        return self

    def render(self, messages: list) -> str:
        """chat 메시지 → 모델별 프롬프트 문자열. Qwen3의 thinking 모드는 끈다."""
        try:
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:                        # enable_thinking 을 모르는 모델
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)

    def reply(self, messages: list, max_new_tokens: int = 220) -> str:
        prompt = self.render(messages)
        encoded = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.inference_mode():
            produced = self.model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                do_sample=False,                 # 재현 가능한 결정적 디코딩
                pad_token_id=self.tokenizer.eos_token_id,
            )
        fresh = produced[0][encoded["input_ids"].shape[1]:]
        return self.tokenizer.decode(fresh, skip_special_tokens=True).strip()

In [ ]:
GENERATOR = LocalGenerator(CFG.generator_id, DEVICE).load()

### 파사드로 묶기

수집 → 검색 → 프롬프트 → 생성을 매번 손으로 이어 붙이는 대신
`Pde5Assistant` 하나에 넣는다. 이렇게 해 두면 질문만 바꿔 가며 실험할 수 있고,
나중에 이 클래스를 그대로 웹 API 뒤에 놓을 수도 있다.

In [ ]:
@dataclass
class GroundedAnswer:
    question: str
    answer: str
    hits: list
    elapsed: float

    def show(self) -> None:
        print(f"Q. {self.question}")
        print("=" * 92)
        print(self.answer)
        print("=" * 92)
        print(f"근거 ({len(self.hits)}건, 생성 {self.elapsed:.1f}초)")
        for rank, hit in enumerate(self.hits, 1):
            print(f"  [S{rank}] {hit.passage.citation():<22} score={hit.score:.4f}")


class Pde5Assistant:
    """검색 + 프롬프트 + 생성을 하나로 묶은 파사드."""

    def __init__(self, index: PassageIndex, encoder: SentenceTransformer,
                 generator: LocalGenerator, config: PipelineConfig):
        self.index = index
        self.encoder = encoder
        self.generator = generator
        self.config = config

    def ask(self, question: str, k: "int | None" = None) -> GroundedAnswer:
        started = time.time()
        hits = self.index.query(question, self.encoder, k=k or self.config.top_k)
        messages = GroundedPrompt.compose(question, hits)
        answer = self.generator.reply(messages, max_new_tokens=self.config.max_new_tokens)
        return GroundedAnswer(question, answer, hits, time.time() - started)


ASSISTANT = Pde5Assistant(RELOADED, ENCODER, GENERATOR, CFG)
print("어시스턴트 준비 완료 —", f"{len(RELOADED)} 패시지 · top-{CFG.top_k}")

In [ ]:
answer_mechanism = ASSISTANT.ask(QUERY_MECHANISM, k=4)
answer_mechanism.show()

In [ ]:
answer_safety = ASSISTANT.ask(QUERY_SAFETY, k=4)
answer_safety.show()

### 대조 실험 — 근거 없이 같은 질문을 던지면

RAG가 실제로 무엇을 바꾸는지 보려면 **같은 모델에게 근거 없이 물어봐야** 한다.
아래는 검색 단계를 건너뛰고 질문만 준 경우다.

In [ ]:
def ask_without_evidence(question: str, max_new_tokens: int = 160) -> str:
    messages = [
        {"role": "system", "content": "You are a biomedical assistant. Be concise: at most six sentences."},
        {"role": "user", "content": question},
    ]
    return GENERATOR.reply(messages, max_new_tokens=max_new_tokens)


bare = ask_without_evidence(QUERY_SAFETY)
print("[근거 없음]")
print(bare)
print()
print("[근거 있음 — 위 RAG 답변 재출력]")
print(answer_safety.answer)

> **무엇을 비교해야 하나.** 두 답변의 *유창함*은 비슷할 것이다. 차이는 다른 데 있다.
>
> - **검증 가능성** — RAG 답변은 `[S1]` 을 따라가면 PDF 몇 쪽인지까지 확인된다.
>   근거 없는 답변은 맞는지 틀리는지 확인할 방법이 없다.
> - **최신성** — 코퍼스는 2026년 논문이다. 모델의 사전학습 지식은 그보다 오래됐다.
> - **범위 인식** — 근거에 없는 것을 물으면 RAG 쪽은 “자료에 없다”고 답할 수 있다.
>   이것이 환각 억제의 실질적 메커니즘이다.

In [ ]:
# 코퍼스가 다루지 않는 질문 — 모델이 한계를 인정하는지 본다
OUT_OF_SCOPE = "What is the recommended tadalafil dose for pediatric pulmonary hypertension?"

probe_hits = RELOADED.query(OUT_OF_SCOPE, ENCODER, k=3)
print("검색 점수:", [round(h.score, 3) for h in probe_hits])
print("최고 점수가 0.55 아래면 근거가 사실상 없다는 신호다.\n")

out_of_scope = ASSISTANT.ask(OUT_OF_SCOPE, k=3)
out_of_scope.show()

---
## 워크북 코드 ↔ 이 노트북 대응표

인쇄된 워크북 4절의 스니펫 `[4-1]~[4-4]` 는 개념을 최소한으로 보여주는 축약 코드다.
이 노트북은 같은 5단계를 **클래스 단위로 재설계**했기 때문에 이름이 다르다.
아래 표로 대응 관계를 확인하면 된다. **동작과 결과는 같다.**

| 워크북 | 이 노트북 | 비고 |
|--------|-----------|------|
| `[4-1]` `import fitz` | `import pymupdf` (실패 시 `fitz` 폴백) | `fitz` 는 구 이름, 현재 deprecated |
| `[4-1]` `nlp = English(); add_pipe('sentencizer')` | `SentenceSplitter` 클래스 내부 | 동일 API를 감싼 것 |
| `[4-1]` `num_sentence_chunk_size = 10` | `CFG.sentences_per_passage = 10` | 같은 값, 설정 dataclass로 이동 |
| — (워크북엔 없음) | `CFG.sentence_stride = 8` | **추가**: 창을 겹쳐 경계 손실 완화 |
| `[4-1]` 짧은 청크 제거 (≤30토큰) | `CFG.min_passage_tokens = 32` + `PassageBuilder` | 임계값만 32로 조정 |
| — (워크북엔 없음) | `looks_like_bibliography()`, `PdfTextLayer._strip_running_lines()` | **추가**: 참고문헌·머리말 제거 |
| `[4-2]` `SentenceTransformer("all-mpnet-base-v2")` | `ENCODER` (`CFG.embedder_id`) | 동일 모델, 768차원 |
| `[4-2]` `text_chunks_and_embeddings_df.csv` | `pde5_index/{vectors.npz, passages.jsonl, manifest.json}` | CSV 대신 벡터/메타 분리 저장 |
| `[4-2]` `pages_and_chunks_over_min_token_len` | `PASSAGES` (`list[Passage]`) | dict 리스트 → dataclass 리스트 |
| `[4-3]` `util.dot_score(a=q, b=embeddings)` | `PassageIndex.query()` 내부 | 동일 호출을 메서드로 감쌈 |
| `[4-3]` `torch.topk(dot_scores, k=5)` | `PassageIndex.query(..., k=CFG.top_k)` | 동일 |
| `[4-3]` `page_number` | `Passage.page_no` (1-base) | 오프셋 보정 없이 PDF 뷰어 쪽번호와 일치 |
| `[4-4]` `model_id = "google/gemma-2b-it"` | `CFG.generator_id = "Qwen/Qwen3-1.7B"` | gemma 는 gated → Qwen3 기본, 교체 가능 |
| `[4-4]` `dialogue_template` | `GroundedPrompt.compose()` | 근거 태그 `[S1]` 추가 |
| `[4-4]` `apply_chat_template(...)` | `LocalGenerator.render()` | `enable_thinking=False` 추가 |
| `[4-4]` `.to('cuda')` | `LocalGenerator._load_kwargs()` | CPU/GPU 자동 분기 |

## 한계와 개선 방향

이 파이프라인은 **작동하는 최소 구성**이다. 실무에 올리려면 다음이 남는다.

| 한계 | 증상 | 개선 방향 |
|------|------|-----------|
| 내적 하나로만 검색 | 동의어·약어(PDE5i vs PDE-5 inhibitor)를 놓침 | BM25 키워드 검색과 **하이브리드**, RRF로 순위 병합 |
| 상위 k를 그대로 사용 | 4위·5위가 노이즈여도 프롬프트에 들어감 | **cross-encoder 재순위화** (`ms-marco-MiniLM`) |
| 전수 비교 (O(N)) | 패시지 수백 개라 괜찮지만 수십만이면 느림 | FAISS·HNSW 근사 최근접 탐색 |
| 표·그림을 못 읽음 | 수치가 표에만 있으면 검색 불가 | 표 전용 추출(`page.find_tables()`) 후 별도 색인 |
| 청크가 문서 구조를 모름 | Methods와 Discussion이 한 창에 섞임 | 섹션 헤딩 기반 분할 |
| 긴 패시지가 절단됨 | mpnet 입력 창은 384토큰 — 초과분은 인코딩에서 버려짐 | 창을 6문장으로 축소하거나 long-context 임베더 사용 |
| 페이지 경계에서 겹침 없음 | 청킹이 페이지 단위라 쪽이 바뀌면 stride가 끊김 | 문서 전체를 이어 붙인 뒤 청킹하고 쪽번호는 오프셋으로 추적 |
| 답변 검증 없음 | 모델이 `[S1]` 을 잘못 인용해도 통과 | 인용 태그 ↔ 근거 문장 일치 자동 검사 |
| 코퍼스 6편 | 질의 범위가 좁음 | Europe PMC 검색 API로 수십~수백 편 확장 |

## 연습 문제

1. **stride 실험** — `CFG.sentence_stride` 를 10(겹침 없음)과 5(절반 겹침)로 바꿔
   패시지 수와 `QUERY_MECHANISM` 의 top-1 점수가 어떻게 변하는지 기록하라.
2. **필터 임계값** — `looks_like_bibliography()` 의 숫자 밀도 임계값 0.11 을 0.05 / 0.20 으로
   바꾸면 몇 개의 패시지가 더 걸러지는가? 본문이 잘못 걸러지는 경우는 없는가?
3. **재순위화 붙이기** — `sentence-transformers` 의 `CrossEncoder` 로 상위 20개를 다시 매겨
   상위 5개로 줄인 뒤, 답변 품질이 달라지는지 비교하라.
4. **코퍼스 확장** — `EPMC_SEARCH` 에 `query=PDE5 AND OPEN_ACCESS:Y` 를 주고
   논문 20편을 자동 수집하도록 `EuropePmcHarvester` 를 확장하라.
   (라이선스 표 출력은 반드시 유지할 것.)
5. **인용 검증기** — 답변의 `[S1]` 태그를 뽑아 실제 그 패시지에 근거가 있는지
   임베딩 유사도로 확인하는 함수를 작성하라.

## 더 알아보기

- **RAG 원논문** — Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)
- **sentence-transformers 공식 문서** — <https://sbert.net/> (의미 검색, 재순위화, CrossEncoder)
- **PyMuPDF 텍스트 추출 가이드** — <https://pymupdf.readthedocs.io/en/latest/recipes-text.html>
- **spaCy sentencizer** — <https://spacy.io/api/sentencizer>
- **Qwen3 모델 카드** — <https://huggingface.co/Qwen/Qwen3-1.7B> (Apache-2.0)
- **Europe PMC REST API** — <https://europepmc.org/RestfulWebService>
- 참고로만: [`mrdbourke/simple-local-rag`](https://github.com/mrdbourke/simple-local-rag) 도
  로컬 RAG를 다루는 공개 예제다. 다만 **2026년 9월 현재 저장소에 오픈소스 라이선스가 선언되어 있지 않으므로**
  (LICENSE 파일 부재, GitHub API `license: null`) 코드를 가져다 쓰거나 파생물을 배포하는 것은 권하지 않는다.
  이 노트북은 해당 저장소의 코드를 사용하지 않았다.

### 다음 실습

- 같은 6일차 [`nb1_scientific_agent.ipynb`](https://github.com/fourmodern/2026_aidrugdiscovery/blob/main/20260905/notebooks/nb1_scientific_agent.ipynb) —
  여기서 만든 것과 같은 “검색”을 **간이 TF-IDF로 축소**한 뒤, 구조화된 기억(LLM-wiki)과 맞세운다.
  이 노트북이 *검색을 잘 만드는 법*이라면, 그쪽은 *검색을 쓸지 기억을 쓸지 고르는 법*이다.
- `T004_nsclc_llm_coscientist.ipynb` — 검색·생성을 여러 에이전트로 나누는 구성.